# Domain 1: Prompting and Task Execution
**Exam**: Claude Certified Associate – Foundations (CCAO-F)
**Weighting**: 14% of exam
**Depth**: deep

## Domain Overview

This domain covers how to turn a business need into an effective Claude prompt: structuring
requests clearly, breaking complex asks into manageable steps, iterating when the first draft
output isn't good enough, and adapting your approach to the type of task (analysis vs. drafting
vs. brainstorming vs. research).

**Why it matters**: the exam tests whether you can go beyond ad-hoc chat messages and reliably
produce useful output on the first or second try, which is the core, everyday skill of a Claude
Associate.

**Key concepts to know**: task decomposition, iterative prompting, prompt structure (role, context,
task, format), and matching prompting style to task type.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Create effective prompts for business and technical tasks

A well-structured business prompt typically states the **role** Claude should take, the
**context** (background info, audience, constraints), the **task** (what exactly to produce), and the
**format** (length, structure, tone). Vague prompts like "write about our Q3 numbers" produce
generic output; specific prompts produce usable output.

In [ ]:
vague_prompt = "Write about our Q3 numbers."

structured_prompt = """
Role: You are a financial communications assistant for a mid-size SaaS company.
Context: Q3 revenue grew 12% QoQ to $4.2M; churn rose from 2.1% to 2.8%; NPS held at 42.
Task: Draft a 150-word summary for the all-hands meeting that highlights the growth,
      names churn as a watch-item, and ends with one forward-looking sentence.
Format: Plain prose, no bullet points, executive tone.
"""

print(call_claude(structured_prompt))

### Apply task decomposition techniques to structure complex requests

Complex requests (e.g., "redesign our onboarding email sequence") are more reliable when
broken into ordered sub-tasks: research/inputs -> draft each component -> review/consolidate.
Decomposition also lets you review and correct intermediate steps before the final output.

In [ ]:
complex_task = "Redesign our 5-email onboarding sequence for a project-management SaaS."

decomposed_steps = [
    "Step 1: List the 5 goals each onboarding email should achieve, in order.",
    "Step 2: For each goal, draft a one-sentence key message.",
    "Step 3: Expand each key message into a full email (subject + 100-150 words).",
    "Step 4: Review all 5 emails together for tone consistency and remove redundancy.",
]

for step in decomposed_steps:
    print("---", step)
    print(call_claude(f"{complex_task}\n\n{step}")[:200], "...\n")

### Iterate prompts to improve output quality

Treat the first response as a draft. Iteration techniques include: giving concrete feedback
("too formal, shorten to 2 sentences"), providing an example of desired output, or asking Claude to
critique its own draft against explicit criteria before revising.

In [ ]:
draft = call_claude("Write a one-paragraph product update for customers about a new export feature.")
print("DRAFT:\n", draft)

feedback_prompt = f"""Here is a draft update:\n{draft}\n
Revise it to: (1) lead with the customer benefit, not the feature name, (2) cut it to 3 sentences,
(3) end with a link placeholder [LINK]."""
revised = call_claude(feedback_prompt)
print("\nREVISED:\n", revised)

### Adapt prompting strategies based on task type

Analysis tasks benefit from asking for explicit criteria and evidence. Research tasks benefit
from asking for sourced, structured findings. Drafting tasks benefit from tone/format constraints.
Brainstorming tasks benefit from asking for a wide, diverse set of options before narrowing down.

In [ ]:
task_type_prompts = {
    "analysis": "Analyze this customer feedback for the top 3 recurring complaints, with evidence quotes: ...",
    "research": "Summarize the current best practices for SaaS trial-to-paid conversion, with sources noted.",
    "drafting": "Draft a formal decline email for a vendor proposal, polite but firm, under 100 words.",
    "brainstorming": "Generate 12 distinct name ideas for a new analytics dashboard feature, one line each.",
}

for task_type, prompt in task_type_prompts.items():
    print(f"[{task_type.upper()}]", call_claude(prompt)[:150], "...\n")

## Mini Hands-On Project

**Covers**: all Domain 1 sub-objectives.

**Scenario**: You're a marketing operations associate. Your manager asks you to produce a
customer-facing blog announcement for a new feature, but only gives you two bullet points of
information. You'll decompose the task, draft with Claude, and iterate to a final version.

### Step 1: Decompose the request

In [ ]:
raw_ask = "Write a blog post announcing our new bulk-export feature. Make it good."
info_bullets = [
    "Bulk export lets users download up to 10,000 records as CSV or JSON in one click.",
    "Available now on the Team and Enterprise plans.",
]

decomposition_prompt = f"""The request is: "{raw_ask}"
Available information: {info_bullets}
List the sub-tasks needed to turn this into a complete, high-quality blog post, in order."""
print(call_claude(decomposition_prompt))

### Step 2: Draft using the decomposed plan

In [ ]:
draft_prompt = f"""Using this information: {info_bullets}
Write a 200-word blog announcement with a headline, one benefit-led paragraph, and a closing CTA."""
draft = call_claude(draft_prompt)
print(draft)

### Step 3: Iterate to final version

In [ ]:
final_prompt = f"""Here is a draft blog post:\n{draft}\n
Revise it to add a one-line quote from a fictional \"VP of Operations\" persona, and shorten the
headline to under 8 words."""
final = call_claude(final_prompt)
print(final)

## Lab Exercises

### Exercise 1: Prompt structure rewrite

**Objective**: Turn a vague prompt into a structured one (role/context/task/format).

**Produce**: A rewritten prompt and Claude's response to it.

In [ ]:
vague = "Help me with my email to the team."

# TODO: rewrite `vague` as a structured prompt with role, context, task, and format sections.
structured = """
Role: ...
Context: ...
Task: ...
Format: ...
"""

print(call_claude(structured))

# Expected output: A structured prompt string and a focused, usable email draft from Claude.

In [ ]:
# STRETCH GOAL: write a small function that validates a prompt string contains
# all four sections (Role, Context, Task, Format) before sending it to call_claude.

def has_all_sections(prompt: str) -> bool:
    # TODO: return True only if all 4 section labels appear in the prompt
    pass

print(has_all_sections(structured))

# Expected output: True, and the function returns False for the original vague prompt.

### Exercise 2: Three-step decomposition

**Objective**: Decompose a complex request into exactly 3 ordered sub-tasks and run each.

**Produce**: Three intermediate outputs and a final combined result.

In [ ]:
complex_request = "Create a customer win-back email campaign for lapsed trial users."

# TODO: define 3 ordered sub-task prompts (list of strings) that build toward the final campaign
sub_tasks = [
    "...",
    "...",
    "...",
]

results = []
for t in sub_tasks:
    results.append(call_claude(t))
    print(results[-1][:150], "...")

# Expected output: A list of 3 outputs, each building on context from the task, ending in a usable campaign email.

In [ ]:
# STRETCH GOAL: feed each result as context into the next sub-task automatically.
running_context = ""
final_results = []
for t in sub_tasks:
    # TODO: prepend `running_context` to each sub-task prompt, then update running_context with the new result
    pass
print(final_results[-1] if final_results else "not implemented")

# Expected output: Each step's prompt includes prior results, producing a more coherent final output than the non-chained version.

### Exercise 3: Feedback-driven iteration

**Objective**: Iterate a single draft twice using specific, actionable feedback.

**Produce**: Three versions of the same text (original, revision 1, revision 2).

In [ ]:
original = call_claude("Write a product FAQ answer for: \"Can I cancel anytime?\"")

# TODO: write feedback text for revision 1 (e.g. tone, length) and generate revision_1
feedback_1 = "..."
revision_1 = call_claude(f"{original}\n\nFeedback: {feedback_1}")

# TODO: write feedback text for revision 2 and generate revision_2
feedback_2 = "..."
revision_2 = call_claude(f"{revision_1}\n\nFeedback: {feedback_2}")

print(original, revision_1, revision_2, sep="\n---\n")

# Expected output: Three progressively refined FAQ answers reflecting each round of feedback.

In [ ]:
# STRETCH GOAL: ask Claude itself to diff the versions and summarize the changes.
diff_prompt = f"""Compare these three FAQ answer versions and list exactly what changed at each step:
V1: {original}
V2: {revision_1}
V3: {revision_2}"""
print(call_claude(diff_prompt))

# Expected output: A short written comparison identifying exactly what changed between each version and why.

### Exercise 4: Task-type matching

**Objective**: Given four unlabeled requests, classify each by task type and adapt the prompt accordingly.

**Produce**: A mapping of request -> task type -> adapted prompt -> output.

In [ ]:
unlabeled_requests = [
    "Why did our support ticket volume spike last week?",
    "What are 3 competitors doing for pricing pages right now?",
    "Write a farewell message for a retiring employee.",
    "Give me 10 ideas for our next webinar topic.",
]

# TODO: for each request, decide task type (analysis/research/drafting/brainstorming)
# and adapt the prompt using techniques appropriate to that type before calling call_claude
for req in unlabeled_requests:
    task_type = "..."  # TODO
    adapted_prompt = req  # TODO: enhance this based on task_type
    print(task_type, "->", call_claude(adapted_prompt)[:120], "...")

# Expected output: Each request correctly typed, with an adapted prompt that reflects that type's best practice.

In [ ]:
# STRETCH GOAL: have Claude itself classify the task type first, then branch your logic on its answer.
for req in unlabeled_requests:
    classification = call_claude(f'Classify this request as one word - analysis, research, drafting, or brainstorming: "{req}"')
    print(classification.strip(), "->", req)

# Expected output: Claude's classifications mostly match your manual classifications from the base exercise.

### Exercise 5: End-to-end business task

**Objective**: Combine decomposition, iteration, and task-type adaptation on one real scenario.

**Produce**: A finished internal memo about a made-up policy change.

In [ ]:
scenario = "Announce a new company policy: Fridays are now meeting-free for focus work."

# TODO: 1) decompose into sub-tasks, 2) draft using an adapted prompt for a \"drafting\" task,
# 3) iterate once using specific feedback, 4) print the final memo
plan = call_claude(f"List the sub-tasks needed to write an internal memo announcing: {scenario}")
print(plan)

draft = call_claude("...")  # TODO
final_memo = call_claude("...")  # TODO
print(final_memo)

# Expected output: A complete, iterated internal memo produced via a visible decompose -> draft -> revise pipeline.

In [ ]:
# STRETCH GOAL: wrap the whole pipeline in a reusable function `business_task_pipeline(scenario: str) -> str`
def business_task_pipeline(scenario: str) -> str:
    # TODO: implement decompose -> draft -> revise using call_claude, return the final text
    pass

print(business_task_pipeline("Announce that the office coffee machine is being upgraded."))

# Expected output: A reusable function that produces a coherent memo for any one-line scenario input.

## Self-Assessment: Prompting and Task Execution

Answer each question, then check the Answers cell below.


**Q1.** Which prompt element is missing if a request states the task and format but gives no background on the audience or constraints?

**Q2.** You ask Claude to "redesign our onboarding flow" in one shot and get a shallow answer. What technique should you apply next?

**Q3.** A first draft is accurate but too long and too casual for an executive audience. What is the most effective next prompting step?

**Q4.** Which task type benefits most from asking for a wide, diverse set of options before narrowing down?

**Q5.** True or False: iterating a prompt means starting over with a completely new prompt each time.

## Answers


**A1.** Context is missing — without it, Claude cannot tailor tone, scope, or relevance to the actual situation.

**A2.** Task decomposition — break the redesign into ordered sub-tasks (current-state analysis, goals, per-step changes) and address them individually.

**A3.** Iterate with specific, actionable feedback (e.g., "shorten to 3 sentences, use a formal tone") rather than re-asking from scratch.

**A4.** Brainstorming — diversity before narrowing is the core value of brainstorming-style prompts.

**A5.** False — iteration builds on the prior output with targeted feedback, rather than discarding it.

---
**Next**: [Domain 2: Output Evaluation and Validation](./02-output-evaluation-and-validation.ipynb)
**Back**: [README](./README.md)